In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://results.tcslondonmarathon.com/2025/?pid=search"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text if soup.title else "No title")

200
TCS London Marathon 2025


In [ ]:
rows = soup.select("li.list-group-item")

clean_rows = []
for row in rows:
    text = row.get_text(" | ", strip=True)
    if "Results" in text or "Place" in text:
        continue
    clean_rows.append(text)

split_rows = [r.split(" | ") for r in clean_rows]

In [48]:
df = pd.DataFrame(split_rows)

df = df[[0, 1, 2, 3, 5, 7, 9, 11, 13, 15]].copy()

df.columns = [
    "place_overall",
    "place_gender",
    "place_category",
    "name",
    "club",
    "runner_number",
    "category",
    "event",
    "half_time",
    "finish_time"
]

df

,place_overall,place_gender,place_category,name,club,runner_number,category,event,half_time,finish_time
0,4979,683,483,"(Hampden) Martin, Minty (Araminta) (GBR)",–,3171,18-39,Mass,01:36:31,03:13:09
1,42125,26099,12881,"-, Sofian (INA)",–,48813,18-39,Mass,02:27:22,05:21:57
2,40660,15194,749,"A Thomas, Deborah (USA)",–,27209,55-59,Mass,02:22:56,05:15:53
3,27750,18697,9449,"A'Bear, Neil (GBR)",–,26140,18-39,Mass,02:00:51,04:32:06
4,55476,24016,3394,"Aarons, Zoe (GBR)",Clapham Chasers,62008,40-44,Mass,03:18:34,07:15:56
5,41909,15867,785,"Aasgaard, Tone (NOR)",–,25645,55-59,Mass,02:33:20,05:21:00
6,5743,4853,870,"Aasheim, Jan Fredrik (NOR)",–,40388,40-44,Mass,01:32:31,03:16:53
7,13348,9876,279,"Ab Elwyn, Rhys (GBR)",Eryri Harriers,39345,60-64,Mass,01:51:31,03:46:40
8,1914,774,43,"Ababovic, Marjana (BIH)",–,V548,60-64,Virtual,–,05:40:29
9,34583,22422,2398,"Abad, Oscar (ESP)",–,11405,50-54,Mass,02:06:55,04:53:59


In [49]:
df = df.replace("–", pd.NA)

cols_num = ["place_overall", "place_gender", "place_category", "runner_number"]
df[cols_num] = df[cols_num].apply(pd.to_numeric, errors="coerce")

df["year"] = 2025

In [50]:
df["half_time_td"] = pd.to_timedelta(df["half_time"], errors="coerce")
df["finish_time_td"] = pd.to_timedelta(df["finish_time"], errors="coerce")

df["half_time_min"] = df["half_time_td"].dt.total_seconds() / 60
df["finish_time_min"] = df["finish_time_td"].dt.total_seconds() / 60

df["second_half_td"] = df["finish_time_td"] - df["half_time_td"]
df["second_half_min"] = df["second_half_td"].dt.total_seconds() / 60

df["split_ratio"] = df["second_half_min"] / df["half_time_min"]

In [51]:
df.head()

,place_overall,place_gender,place_category,name,club,runner_number,category,event,half_time,finish_time,year,half_time_td,finish_time_td,half_time_min,finish_time_min,second_half_td,second_half_min,split_ratio
0,4979.0,683.0,483.0,"(Hampden) Martin, Minty (Araminta) (GBR)",NaN,3171.0,18-39,Mass,01:36:31,03:13:09,2025,0 days 01:36:31,0 days 03:13:09,96.516667,193.150000,0 days 01:36:38,96.633333,1.001209
1,42125.0,26099.0,12881.0,"-, Sofian (INA)",NaN,48813.0,18-39,Mass,02:27:22,05:21:57,2025,0 days 02:27:22,0 days 05:21:57,147.366667,321.950000,0 days 02:54:35,174.583333,1.184687
2,40660.0,15194.0,749.0,"A Thomas, Deborah (USA)",NaN,27209.0,55-59,Mass,02:22:56,05:15:53,2025,0 days 02:22:56,0 days 05:15:53,142.933333,315.883333,0 days 02:52:57,172.950000,1.210005
3,27750.0,18697.0,9449.0,"A'Bear, Neil (GBR)",NaN,26140.0,18-39,Mass,02:00:51,04:32:06,2025,0 days 02:00:51,0 days 04:32:06,120.850000,272.100000,0 days 02:31:15,151.250000,1.251552
4,55476.0,24016.0,3394.0,"Aarons, Zoe (GBR)",Clapham Chasers,62008.0,40-44,Mass,03:18:34,07:15:56,2025,0 days 03:18:34,0 days 07:15:56,198.566667,435.933333,0 days 03:57:22,237.366667,1.195400


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype          
---  ------           --------------  -----          
 0   place_overall    24 non-null     float64        
 1   place_gender     24 non-null     float64        
 2   place_category   24 non-null     float64        
 3   name             25 non-null     str            
 4   club             4 non-null      str            
 5   runner_number    21 non-null     float64        
 6   category         25 non-null     str            
 7   event            25 non-null     str            
 8   half_time        21 non-null     str            
 9   finish_time      24 non-null     str            
 10  year             25 non-null     int64          
 11  half_time_td     21 non-null     timedelta64[us]
 12  finish_time_td   24 non-null     timedelta64[us]
 13  half_time_min    21 non-null     float64        
 14  finish_time_min  24 non-null     float6

In [60]:
df.to_csv("../data/raw/london_2025_results.csv", index=False)